# 03 · Write your own entity definition

An **entity definition** is a YAML file that says two things:

1. **what the LLM should find** (`name`, `definition`, child entities), and
2. **where each piece goes in an OMOP table** (the `omop` block, never sent to the model).

This example builds body weight with its unit for the `measurement` table. No terminology
mapper is needed: the concept is fixed and units are looked up as UCUM codes.

In [1]:
# Run from the folder where you ran `aatm init` (it holds .aatm/ and .env).
import os

if not os.path.isdir(".aatm") and os.path.isdir("../.aatm"):
    os.chdir("..")

## 1. The definition

The file is [definitions/body_weight.yaml](definitions/body_weight.yaml). The value and
the unit are **children** of the weight mention, so the `omop` block reads them with
`from: child.<key>`.

In [2]:
from pathlib import Path

path = Path("examples/definitions/body_weight.yaml")
print(path.read_text())

# Body weight, stored in the OMOP measurement table with its unit.
# Used by examples/03_custom_entity_definition.ipynb.

- name: Body weight
  key: body_weight
  definition: >
    The patient's measured body weight: the word for weight, the value and the unit,
    copied as written, e.g. "weight 72 kg" or "Wt: 158 lb". Stop at the unit: no
    words or punctuation after it. Does not include target or ideal weights, weight
    changes such as "lost 5 kg", or the weight of anyone other than the patient.
  omop:
    table: measurement
    fields:
      person_id: {from: context.person_id}                # from the ClinicalNote
      measurement_concept_id: {concept: "LOINC:29463-7"}   # Body weight, always the same
      measurement_date: {from: context.note_date}
      measurement_type_concept_id: {value: 32858}          # NLP
      measurement_source_value: {from: text}               # the words of the mention
      value_as_number: {from: child.weight_value, as: float}
      unit_conc

Each column takes one source:

| Source | Value |
|---|---|
| `{from: text}` | the words of the mention |
| `{from: child.<key>}` / `{from: attribute.<key>}` | a child's text / an attribute's label |
| `{from: context.<field>}` | a field of the `ClinicalNote` |
| `{value: 32858}` / `{concept: "LOINC:29463-7"}` | a constant / the concept id of a code |

Options such as `as: float`, `lookup: UCUM` or `map: {}` (notebook 04) convert or code the
value. See the [full reference](https://precisiondata.github.io/aatm/getting-started/omop_tables/#reference-the-omop-block).

## 2. Check it before spending model calls

Loading the file and creating the builder validate column names, `child.` keys, codes,
standard concepts and domains. `describe_concepts()` lists every fixed code:

In [3]:
from aatm.data_models.extractors import EntityDefinitions
from aatm.omop import OmopTableBuilder

definitions = EntityDefinitions.from_yaml(path)
OmopTableBuilder(definitions).describe_concepts()

,entity,column,label,code,concept_id,concept_name,vocabulary_id,domain_id
0,body_weight,measurement_concept_id,None,LOINC:29463-7,3025315,Body weight,LOINC,Measurement


A typo fails right away, with a message saying what to fix:

In [4]:
import tempfile

broken = Path(tempfile.mkdtemp()) / "broken.yaml"
broken.write_text(
    path.read_text().replace("LOINC:29463-7", "LOINC:29463-8")
)  # a typo in the code

try:
    OmopTableBuilder(EntityDefinitions.from_yaml(broken))
except ValueError as error:
    print(error)

Concepts not found in the vocabulary: ['LOINC:29463-8']


This is the prompt the model reads (no `omop` block in it):

In [5]:
print(definitions.to_prompt_str())

Body weight (key: `body_weight`, has_parent: False, parent_key: `None`): The patient's measured body weight: the word for weight, the value and the unit, copied as written, e.g. "weight 72 kg" or "Wt: 158 lb". Stop at the unit: no words or punctuation after it. Does not include target or ideal weights, weight changes such as "lost 5 kg", or the weight of anyone other than the patient.
Weight value (key: `weight_value`, has_parent: True, parent_key: `body_weight`): The number of a body weight, without the unit, e.g. "72" from "weight 72 kg".
Weight unit (key: `weight_unit`, has_parent: True, parent_key: `body_weight`): The unit of a body weight, copied as written without punctuation, e.g. "kg" or "lb".


## 3. Run it

In [6]:
from aatm.omop import ClinicalNote, OmopExtractionPipeline

notes = [
    ClinicalNote(
        text="Weight 72 kg today. Goal weight 65 kg.",
        person_id=1,
        note_date="2025-06-01",
    ),
    ClinicalNote(
        text="Wt: 158 lb. Lost 5 lb since March.", person_id=2, note_date="2025-06-02"
    ),
]

pipeline = OmopExtractionPipeline(definitions, model_id="openai/gpt-5-mini")
tables = await pipeline(notes)
tables["measurement"].dropna(axis=1, how="all")

,measurement_id,person_id,measurement_concept_id,measurement_date,measurement_type_concept_id,value_as_number,unit_concept_id,measurement_source_value,unit_source_value
0,1,1,3025315,2025-06-01,32858,72.0,9529,Weight 72 kg,kg
1,2,2,3025315,2025-06-02,32858,158.0,0,Wt: 158 lb,lb


The goal weight and the weight loss were left out, as the definition asks.

## 4. Read the issues

In [7]:
tables.issues_dataframe()[["row_id", "column", "reason", "detail"]]

,row_id,column,reason,detail
0,2,unit_concept_id,unmapped,No concept for 'lb' with a UCUM code lookup; s...


"kg" is a UCUM code (*kilogram*, 9529). "lb" is not (UCUM writes `[lb_av]`), so the row
keeps `unit_concept_id = 0` ("there is a unit, but it is not mapped") and
`unit_source_value = "lb"`.

**Tip:** most fixes are in the wording of `definition`. Iterate on real notes and re-read
`to_prompt_str()`.

## Next

Code free text with the terminology mapper: [04_mapper_configuration.ipynb](04_mapper_configuration.ipynb).